In [ ]:
# Quantum simulation libraries
from qutip import (
    basis, 
    mesolve, 
    qeye, 
    sigmax, 
    sigmay, 
    sigmaz, 
    tensor,
    )
import qutip

# Machine learning libraries
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

# Plotting libraries
import matplotlib.pyplot as plt
from rich.progress import track
import seaborn as sns

# Linalg libraries
import numpy as np
from scipy.stats import pearsonr
import scipy.interpolate

# Helper libraries
from dataclasses import dataclass

# Data libraries
import yfinance as yf


In [ ]:
# set the device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

# Load target data

In [ ]:
dow = yf.Ticker("DOW")
history = dow.history(period="max")

In [ ]:
history

In [ ]:
plt.plot(history["Open"].to_numpy())

In [ ]:
class BFieldGenerator:
    def __init__(
        self,  raw_data: np.ndarray, times: np.ndarray
        ):
        """
        Create the BField generator.

        Parameters
        ----------
        amplitude : float
            Amplitude of the magnetic field.
        frequency : float
            Frequency of the cosine wave.
        resoltion : float
            Distance between measured points.
        length : int
            Amount of time to run it for.
        """
        self.raw_data = raw_data

        self.counter = 0

        self.measured_field = []
        self.measured_times = []
        self.times = times

    def __call__(self, t: float):
        """
        Get the next value of the magnetic field.

        Parameters
        ----------
        t : float
            Current time.
        """
        if t == self.times[self.counter]:
            driving_field = self.raw_data[self.counter]

            self.measured_field.append(driving_field)      
            self.measured_times.append(t)
            self.counter += 1
        else:
            # try:
            driving_field = self.measured_field[-1]
            # except IndexError:
            #     driving_field = self.raw_data[0]

        return np.array(driving_field)

# Run Simulation

In [ ]:
@dataclass
class SimulationParameters:
    """
    Helper class for the simulation parameters.
    """
    length: int
    coupling: list

@dataclass
class SimulationState:
    """ 
    Helper class for the simulation state.
    """
    number_of_spins: int
    quantum_state: list
    spin_list: list
    coupling_list: list

In [ ]:
def get_simulation_state(parameters: SimulationParameters):
    """
    Returns the initial state of the simulation.
    """
    # Get the initial wavefunction
    number_of_spins = parameters.length

    initial_state = [basis(2, 1)] + [basis(2, 0)] * (number_of_spins - 1)

    # Setup operators for individual qubits
    sx_list, sy_list, sz_list = [], [], []
    for i in range(number_of_spins):
        op_list = [qeye(2)] * number_of_spins
        op_list[i] = sigmax()
        sx_list.append(tensor(op_list))
        op_list[i] = sigmay()
        sy_list.append(tensor(op_list))
        op_list[i] = sigmaz()
        sz_list.append(tensor(op_list))

    # Setup the operators for the coupling
    Jx = parameters.coupling * np.ones(number_of_spins)
    Jy = parameters.coupling * np.ones(number_of_spins)
    Jz = parameters.coupling * np.ones(number_of_spins)    

    return SimulationState(
        number_of_spins=number_of_spins,
        quantum_state=tensor(initial_state),
        spin_list=[sx_list, sy_list, sz_list],
        coupling_list=[Jx, Jy, Jz],
    )

In [ ]:
def compute_hamiltonian(t, args):
    """
    Compute the Hamiltonian at time t.
    
    Parameters
    ----------
    t : float
        Current time.
    args : dict
        System parameters in the H computation.
    """
    sx_list = args["sx_list"]
    sy_list = args["sy_list"]
    sz_list = args["sz_list"]
    Jx = args["Jx"]
    Jy = args["Jy"]
    Jz = args["Jz"]
    driving_field = args["driving"]

    N = args["N"]

    # Magnetic field terms to top row
    H = 0
    for i in [0, 1, 3]:
        H -= float(driving_field(t)) * sz_list[i]

    
    # Interaction terms
    for n in range(N - 1):
        H += -0.5 * Jx[n] * sx_list[n] * sx_list[n + 1]
        H += -0.5 * Jy[n] * sy_list[n] * sy_list[n + 1]
        H += -0.5 * Jz[n] * sz_list[n] * sz_list[n + 1]

    return H

In [ ]:
price_data = history["Open"].to_numpy()
price_data = price_data / np.max(price_data)
times = np.linspace(0, 10, price_data.shape[0])

In [ ]:
# for strength in np.linspace(1, 10, 10, dtype=int):
simulation_parameters = SimulationParameters(
    length=5,
    coupling=2.0 * np.pi,
)
simulation_state = get_simulation_state(simulation_parameters)

field_generator = BFieldGenerator(price_data, times=times)
args = {
    "sx_list": simulation_state.spin_list[0],
    "sy_list": simulation_state.spin_list[1],
    "sz_list": simulation_state.spin_list[2],
    "Jx": simulation_state.coupling_list[0],
    "Jy": simulation_state.coupling_list[1],
    "Jz": simulation_state.coupling_list[2],
    "N": simulation_state.number_of_spins,
    "driving": field_generator
}

In [ ]:
results = mesolve(compute_hamiltonian, simulation_state.quantum_state, times, [], [], args)

# qutip.qsave(states, f"{strength}")

In [ ]:
measured_field = np.array(field_generator.measured_field)

In [ ]:
fit_generator = BFieldGenerator(price_data, times=times)
b_field = []
for t in times:
    b_field.append(fit_generator(t))

b_field = np.array(b_field)

In [ ]:
price_data

In [ ]:
b_field

In [ ]:
measured_field.shape

In [ ]:
np.shape(results.times)

In [ ]:
field_generator.measured_times[-1]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 10))

ax[0].plot(results.times, b_field)
ax[0].set_xlabel("Time")
ax[0].set_ylabel("B Field")

ax[1].plot(results.times[1:], measured_field)
ax[1].set_xlabel("Time")
ax[1].set_ylabel("B Field")

plt.show()

In [ ]:
state_dimension = 100

measurements = []

for _ in range(state_dimension):
    non_gue_matrix = qutip.rand_herm(
        32, 
        0.9, 
        dims=[[2, 2, 2, 2, 2], [2, 2, 2, 2, 2]]
    )
    measurements.append(non_gue_matrix)

In [ ]:
# Compute observables
observations = np.zeros((measured_field.shape[0], state_dimension))
states = results.states[1:]

for t, state in enumerate(states):
    for o, operator in enumerate(measurements):
        measure_state = state
        observations[t][o] = qutip.expect(measure_state * measure_state.dag(), operator)

# Fit Model

In [ ]:
class NeuralNetwork(nn.Module):
    
    def __init__(self, state_dimension: int, output_dimension: int):
        """
        Build the network.
        
        Parameters
        ----------
        state_dimension : int
                Dimension of the state representation.
                This is the input to the layer.
        output_dimension : int
                Dimension of the output being predicted.
        """
        super().__init__()

        self.linear_stack = nn.Sequential(
            nn.Linear(state_dimension, 128),
            nn.ReLU(),
            nn.Linear(128, output_dimension)
        )
    
    def forward(self, x):
        """
        Forward pass through the network.
        
        As we are doing reservoir computing, this is 
        simply a linear layer.
        """
        return self.linear_stack(x)

In [ ]:
class ReservoirDataset(Dataset):
    """
    Custom dataset for the training.
    """
    def __init__(
        self, 
        state_data: np.ndarray, 
        function_data: np.ndarray,
    ):
        """
        Constructor for the dataset.

        Parameters
        ----------
        state_data : np.ndarray
                State description data.
        function_data : np.ndarray
                Function data being fit.
                This will be the target data.
        prediction_length : int
                How far into the future you will predict.
        """
        self.state_data = torch.Tensor(state_data).to(device)

        self.function_data = torch.Tensor(
            function_data.reshape(-1, 1)
        ).to(device)
        
        self.norm_factor = 1 # max(abs(function_data.flatten()))

    def split_data(self, train_size: float):
        """
        Split the data into training and validation sets.
        
        Parameters
        ----------
        train_size : float
                Size of the training set.
        """
        train_size = int(train_size * len(self))
        val_size = len(self) - train_size
        return torch.utils.data.random_split(self, [train_size, val_size])
    
    def __len__(self):
        """
        Length of the dataset.
        """
        return int(
            len(self.function_data)
        )
    
    def __getitem__(self, idx: int):
        """
        Collect an item from the dataset.
        
        Parameters
        ----------
        idx : int
                Index of the state to take.
        """
        state = self.state_data[idx]
        target = self.function_data[idx] / self.norm_factor
        
        return state, target

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss = loss.item()
            
    return loss

In [ ]:
def test(dataloader, model, loss_fn) -> np.ndarray:
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
    test_loss /= num_batches
    
    return test_loss

In [ ]:
def train_model(dataset, test_ds, model = None):
    """
    Train a model on the current data.
    """    
    if model is None:
        model = NeuralNetwork(
            state_dimension=100,
            output_dimension=1
        ).to(device)

        model = model.type(torch.float32)

    # Use MSE loss
    loss_fn = nn.MSELoss()

    # Use ADAM optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    # Create the loader
    loader = DataLoader(dataset, batch_size=250, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=250, shuffle=False)
    
    # Train the network
    epochs = 3000
    train_losses = []
    test_losses = []

    for t in track(range(epochs)):
        loss = train(loader, model, loss_fn, optimizer)
        train_losses.append(loss)
        loss = test(test_loader, model, loss_fn)
        test_losses.append(loss)
    try:
        train_losses = [item.cpu().detach().numpy() for item in train_losses]
    except AttributeError:
        pass
    
    return train_losses, test_losses, model

## Train the model

In [ ]:
def train_and_analyse(fraction: float):
    split_fraction = fraction
    train_indices = np.random.choice(
        np.arange(0, history.shape[0] - 1),
        size=int((history.shape[0] - 1) * split_fraction),
        replace=False
    )
    test_indices = np.setdiff1d(np.arange(0, history.shape[0] - 1), train_indices)

    train_state_data = observations[train_indices, :]
    train_function_data = b_field[train_indices]

    test_state_data = observations[test_indices, :]
    test_function_data = b_field[test_indices]

    train_ds = ReservoirDataset(
        state_data=train_state_data,
        function_data=train_function_data,
    )

    test_ds = ReservoirDataset(
        state_data=test_state_data,
        function_data=test_function_data,
    )

    train_losses, test_losses, model = train_model(train_ds, test_ds, model=None)

    test_predictions = []
    test_targets = []
    for item in test_ds:
        state, target = item
        test_predictions.append(model(state).cpu().detach().numpy())
        test_targets.append(target.cpu().detach().numpy())

    test_predictions = np.array(test_predictions).reshape(-1)
    test_targets = np.array(test_targets).reshape(-1)

    train_predictions = []
    train_targets = []
    for item in train_ds:
        state, target = item
        train_predictions.append(model(state).cpu().detach().numpy())
        train_targets.append(target.cpu().detach().numpy())

    train_predictions = np.array(train_predictions).reshape(-1)
    train_targets = np.array(train_targets).reshape(-1)

    test_pearson = pearsonr(test_targets, test_predictions)[0]
    train_pearson = pearsonr(train_targets, train_predictions)[0]

    return train_losses, test_losses, train_pearson, test_pearson

In [ ]:
fractions = np.linspace(0.1, 0.9, 9)
ensembles = 10

train_losses = {}
test_losses = {}
train_correlations = {}
test_correlations = {}

for fraction in fractions:
    train_losses[fraction] = []
    test_losses[fraction] = []
    train_correlations[fraction] = []
    test_correlations[fraction] = []

    for _ in range(ensembles):
        train_loss, test_loss, train_correlation, test_correlation = train_and_analyse(fraction)

        train_losses[fraction].append(train_loss)
        test_losses[fraction].append(test_loss)
        train_correlations[fraction].append(train_correlation)
        test_correlations[fraction].append(test_correlation)

In [ ]:
np.save("train_losses.npy", train_losses)
np.save("test_losses.npy", test_losses)
np.save("train_correlations.npy", train_correlations)
np.save("test_correlations.npy", test_correlations)

In [ ]:
cubic_spline = np.load("cubic-spline.npy", allow_pickle=True).item()
cubic_hermite = np.load("cubic-hermite-spline.npy", allow_pickle=True).item()

# Plot the test correlations
x_data = test_correlations.keys()
y_data = [np.mean(correlations) for correlations in test_correlations.values()]
err_data = [np.std(correlations) for correlations in test_correlations.values()]

plt.errorbar(x_data, y_data, yerr=err_data, fmt="o", mfc="none", capsize=5, label="qRC")
plt.errorbar(
    cubic_spline.keys(),
    [np.mean(correlations) for correlations in cubic_spline.values()],
    yerr=[np.std(correlations) for correlations in cubic_spline.values()],
    fmt="o",
    mfc="none",
    capsize=5,
    label="Cubic Spline",
)
plt.errorbar(
    cubic_hermite.keys(),
    [np.mean(correlations) for correlations in cubic_hermite.values()],
    yerr=[np.std(correlations) for correlations in cubic_hermite.values()],
    fmt="o",
    mfc="none",
    capsize=5,
    label="Cubic Hermite Spline",
)
plt.xlabel("Training Fraction")
plt.ylabel("Pearson Correlation")
plt.legend()

# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
sns.histplot(cubic_spline[0.1], label="Cubic Spline", kde=True, stat="probability")
sns.histplot(cubic_hermite[0.1], label="Cubic Spline", kde=True, stat="probability")
sns.histplot(test_correlations[0.1], label="qRC", kde=True, stat="probability")

In [ ]:
history.shape

In [ ]:
interp_test = {}

for fraction in fractions:
    interp_test[fraction] = []
    for i in range(1000):
        train_indices = np.random.choice(
                np.arange(0, history.shape[0] - 1),
                size=int((history.shape[0] - 1) * fraction),
                replace=False
            )
        test_indices = np.setdiff1d(np.arange(0, history.shape[0] - 1), train_indices)

        train_price_data = price_data[train_indices]
        train_times = times[train_indices]

        test_price_data = price_data[test_indices]
        test_times = times[test_indices]
        sort_indices = np.argsort(train_times)
        # interpolator = scipy.interpolate.interp1d(train_times, train_price_data, kind="cubic", fill_value="extrapolate")
        interpolator = scipy.interpolate.CubicHermiteSpline(train_times[sort_indices], train_price_data[sort_indices], np.gradient(train_price_data[sort_indices], train_times[sort_indices]), extrapolate=True)

        interpolated_price_data = interpolator(test_times)

        interp_test[fraction].append(pearsonr(interpolated_price_data, test_price_data)[0])


In [ ]:
np.save("cubic-hermite-spline.npy", interp_test)

In [ ]:
split_fraction = 0.7
train_indices = np.random.choice(
    np.arange(0, measured_field.shape[0] - 1),
    size=int((measured_field.shape[0] - 1) * split_fraction),
    replace=False
)
test_indices = np.setdiff1d(np.arange(0, history.shape[0] - 1), train_indices)

train_state_data = observations[train_indices, :]
train_function_data = measured_field[train_indices]

test_state_data = observations[test_indices, :]
test_function_data = measured_field[test_indices]

train_ds = ReservoirDataset(
    state_data=train_state_data,
    function_data=train_function_data,
)

test_ds = ReservoirDataset(
    state_data=test_state_data,
    function_data=test_function_data,
)

train_losses, test_losses, model = train_model(train_ds, test_ds, model=None)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].plot(test_losses)
ax[1].plot(train_losses)

ax[1].set_yscale("log")
ax[0].set_yscale("log")

plt.show()

In [ ]:
test_predictions = []
test_targets = []
for item in test_ds:
    state, target = item
    test_predictions.append(model(state).cpu().detach().numpy())
    test_targets.append(target.cpu().detach().numpy())

test_predictions = np.array(test_predictions).reshape(-1)
test_targets = np.array(test_targets).reshape(-1)

train_predictions = []
train_targets = []
for item in train_ds:
    state, target = item
    train_predictions.append(model(state).cpu().detach().numpy())
    train_targets.append(target.cpu().detach().numpy())

train_predictions = np.array(train_predictions).reshape(-1)
train_targets = np.array(train_targets).reshape(-1)

In [ ]:
plt.plot(results.times, b_field, label="baseline")

plt.plot(
    results.times[train_indices], 
    train_predictions, 
    "o",
    label="train", 
    mfc="none", 
    markersize=2
)

plt.plot(
    results.times[test_indices], 
    test_predictions, 
    "o",
    label="test", 
    mfc="none", 
    markersize=2
)
plt.tight_layout()
plt.legend()
plt.xlabel("Time")
plt.ylabel("B Field")

plt.savefig("dow30-70-30.pdf")
plt.show()


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].plot(test_targets, test_predictions, "k.", label=pearsonr(test_targets, test_predictions)[0])
ax[0].plot(test_targets, test_targets, "r--")
ax[0].legend()

ax[1].plot(train_targets, train_predictions, "k.", label=pearsonr(train_targets, train_predictions)[0])
ax[1].plot(train_targets, train_targets, "r--")
ax[1].legend()

# fig.legend()
plt.show()

# Large Image

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

ax[0].plot(test_targets, test_predictions, "k.", label=pearsonr(test_targets, test_predictions)[0])
ax[0].plot(test_targets, test_targets, "r--")
ax[0].legend()

ax[1].plot(train_targets, train_predictions, "k.", label=pearsonr(train_targets, train_predictions)[0])
ax[1].plot(train_targets, train_targets, "r--")
ax[1].legend()

# fig.legend()
plt.show()

In [ ]:
plt.plot(results.times, b_field, label="baseline")

plt.plot(
    results.times[train_indices], 
    train_predictions, 
    "o",
    label="train", 
    mfc="none", 
    markersize=2
)

plt.plot(
    results.times[test_indices], 
    test_predictions, 
    "o",
    label="test", 
    mfc="none", 
    markersize=2
)
plt.tight_layout()
plt.legend()
plt.xlabel("Time")
plt.ylabel("B Field")

# plt.savefig("dow30-70-30.pdf")
plt.show()


In [ ]:
cubic_spline = np.load("cubic-spline.npy", allow_pickle=True).item()
cubic_hermite = np.load("cubic-hermite-spline.npy", allow_pickle=True).item()
test_correlations = np.load("test_correlations.npy", allow_pickle=True).item()
train_correlations = np.load("train_correlations.npy", allow_pickle=True).item()

# Plot the test correlations
x_data = test_correlations.keys()
y_data = [np.mean(correlations) for correlations in test_correlations.values()]
err_data = [np.std(correlations) for correlations in test_correlations.values()]

plt.errorbar(x_data, y_data, yerr=err_data, fmt="o", mfc="none", capsize=5, label="qRC")
plt.errorbar(
    cubic_spline.keys(),
    [np.mean(correlations) for correlations in cubic_spline.values()],
    yerr=[np.std(correlations) for correlations in cubic_spline.values()],
    fmt="o",
    mfc="none",
    capsize=5,
    label="Cubic Spline",
)
plt.errorbar(
    cubic_hermite.keys(),
    [np.mean(correlations) for correlations in cubic_hermite.values()],
    yerr=[np.std(correlations) for correlations in cubic_hermite.values()],
    fmt="o",
    mfc="none",
    capsize=5,
    label="Cubic Hermite Spline",
)
plt.xlabel("Training Fraction")
plt.ylabel("Pearson Correlation")
plt.legend()

# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Create figure
fig = plt.figure(figsize=(12, 6))

# Create 2x3 grid for layout
gs = gridspec.GridSpec(2, 3)

# Large subplot - first column, spans two rows
ax1 = plt.subplot(gs[:, 0])
ax1.plot(results.times, b_field, label="baseline")

ax1.plot(
    results.times[train_indices], 
    train_predictions, 
    "o",
    label="train", 
    mfc="none", 
    markersize=2
)

ax1.plot(
    results.times[test_indices], 
    test_predictions, 
    "o",
    label="test", 
    mfc="none", 
    markersize=2
)
ax1.legend()
ax1.set_xlabel("Time")
ax1.set_ylabel("B Field")

# Upper right subplot
ax2 = plt.subplot(gs[0, 1:])
cubic_spline = np.load("cubic-spline.npy", allow_pickle=True).item()
cubic_hermite = np.load("cubic-hermite-spline.npy", allow_pickle=True).item()
test_correlations = np.load("test_correlations.npy", allow_pickle=True).item()
train_correlations = np.load("train_correlations.npy", allow_pickle=True).item()

# Plot the test correlations
x_data = test_correlations.keys()
y_data = [np.mean(correlations) for correlations in test_correlations.values()]
err_data = [np.std(correlations) for correlations in test_correlations.values()]

ax2.errorbar(x_data, y_data, yerr=err_data, fmt="o", mfc="none", capsize=5, label="qRC")
ax2.errorbar(
    cubic_spline.keys(),
    [np.mean(correlations) for correlations in cubic_spline.values()],
    yerr=[np.std(correlations) for correlations in cubic_spline.values()],
    fmt="o",
    mfc="none",
    capsize=5,
    label="Cubic Spline",
)
ax2.errorbar(
    cubic_hermite.keys(),
    [np.mean(correlations) for correlations in cubic_hermite.values()],
    yerr=[np.std(correlations) for correlations in cubic_hermite.values()],
    fmt="o",
    mfc="none",
    capsize=5,
    label="Cubic Hermite Spline",
)
ax2.set_xlabel("Training Fraction")
ax2.set_ylabel("Pearson Correlation")
ax2.legend()

# Lower right subplot
ax3 = plt.subplot(gs[1, 1])
ax3.plot(test_targets, test_predictions, "k.", label=f"{pearsonr(test_targets, test_predictions)[0]:.3f}")
ax3.plot(test_targets, test_targets, "r--")
ax3.set_xlabel("Target")
ax3.set_ylabel("Prediction")
ax3.legend()

ax4 = plt.subplot(gs[1, 2])
ax4.plot(train_targets, train_predictions, "k.", label=f"{pearsonr(train_targets, train_predictions)[0]:.3f}")
ax4.plot(train_targets, train_targets, "r--")
ax4.set_xlabel("Target")
ax4.set_ylabel("Prediction")
ax4.legend()

# Adjust layout
plt.tight_layout()
plt.savefig("dow30-combo.pdf")
plt.show()